# Análise Espacial Avançada

## 1. Mapas de Isócronas

In [ ]:
import os, json, osmnx as ox, folium, geopandas as gpd, math
data_dir = '../data'
G = ox.load_graphml(os.path.join(data_dir, 'quixada_drive.graphml'))
center_node = list(G.nodes)[0]
center_point = (G.nodes[center_node]['y'], G.nodes[center_node]['x'])

def iso_polygon(minutes):
    meters = minutes * 60 * 5
    def haversine(p1, p2):
        R = 6371000
        lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
        lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    nodes = [n for n in G.nodes if haversine((G.nodes[n]['y'], G.nodes[n]['x']), center_point) <= meters]
    points = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in nodes]
    return gpd.GeoSeries(gpd.points_from_xy([p[1] for p in points], [p[0] for p in points])).union_all().convex_hull

m_iso = folium.Map(location=center_point, zoom_start=14, tiles='OpenStreetMap')
# draw base road network
for u, v, d in G.edges(data=True):
    folium.PolyLine(locations=[(G.nodes[u]['y'], G.nodes[u]['x']),
                       (G.nodes[v]['y'], G.nodes[v]['x'])],
                    color='gray', weight=1, opacity=0.5).add_to(m_iso)




colors = ['#ffeda0', '#feb24c', '#f03b20']
for i, mins in enumerate([5, 10, 15]):
    poly = iso_polygon(mins)
    folium.GeoJson(poly,
        style_function=lambda x, c=colors[i]: {'fillColor': c, 'color': c,
                                              'weight': 2, 'fillOpacity': 0.3}
    ).add_to(m_iso)

m_iso.save(os.path.join(data_dir, 'spatial_analysis_map.html'))
m_iso

## 2. Heatmap de Densidade Viária

In [ ]:
from folium.plugins import HeatMap
coords = [(data['y'], data['x']) for _, data in G.nodes(data=True)]
heat = folium.Map(location=center_point, zoom_start=14, tiles='OpenStreetMap')




HeatMap(coords,
        radius=12,
        blur=8,
        max_zoom=1,
        gradient={0.2:'blue',0.4:'lime',0.6:'orange',0.8:'red'}).add_to(heat)
heat.save(os.path.join(data_dir, 'spatial_analysis_heatmap.html'))
heat